### Row Filters & Column Masks (Filtros de Filas y Máscaras de Columnas)
Este cuaderno muestra cómo implementar la seguridad a nivel de datos directamente sobre una tabla mediante filtros de fila y máscaras de columna.

A diferencia de las vistas dinámicas, en las que la lógica de seguridad se define dentro de una vista, este método permite asociar las reglas de protección directamente a la tabla. De esta manera, las políticas se aplican automáticamente cada vez que se consulta la tabla.

La implementación contempla dos funcionalidades principales:

- **Seguridad a nivel de fila:** determina qué registros puede visualizar cada usuario.

- **Enmascaramiento a nivel de columna:** controla cómo se muestran los valores que contienen información confidencial.

Se utilizá un conjunto de datos de **Clientes** sencillo que permite observar cómo distintos usuarios obtienen resultados diferentes al consultar una misma tabla.

#### Paso 1: Usar el schema "data_protection" en el catalog "demo"

In [0]:
USE CATALOG demo;
USE SCHEMA data_protection;

#### Paso 2: Crear la tabla "customers_privacy"
Se crea una tabla que almacena información de clientes procedentes de distintos países.

In [0]:
CREATE OR REPLACE TABLE customers_privacy (
    customer_id INT,
    name STRING,
    country STRING,
    email STRING,
    balance DECIMAL(10,2),
    credit_card STRING
);

#### Paso 3: Insertar datos de muestra
El conjunto de datos contiene registros correspondientes a los países de Perú y Estados Unidos.

Esto permite comprobar el funcionamiento de la seguridad a nivel de fila, verificando que cada usuario solo pueda visualizar los datos correspondientes a su región.

In [0]:
INSERT INTO customers_privacy VALUES
(2001, 'Carlos', 'Perú', 'carlos@email.com', 8500.00, '4532123456789012'),
(2002, 'Ana', 'Perú', 'ana@email.com', 6200.00, '5214789632147852'),
(2003, 'John', 'USA', 'john@email.com', 9300.00, '6011122233334444'),
(2004, 'Isabel', 'USA', 'isabel@email.com', 9900.00, '6011122233335555'),
(2005, 'Luci', 'Mexico', 'maria@email.com',  4100.00, '3782822463100050');

In [0]:
SELECT * FROM customers_privacy;

#### Paso 4: Crear la Función Row Filter
La función comprueba a qué grupo pertenece el usuario y devuelve una condición que determina si una fila debe incluirse en los resultados de la consulta.

Por ejemplo:
- Los usuarios del grupo Perú pueden visualizar los registros correspondientes a Perú.
- Los usuarios del grupo USA pueden consultar los registros correspondientes a Estados Unidos.
- Los usuarios del grupo Admin tienen permiso para visualizar todos los registros.

In [0]:
CREATE OR REPLACE FUNCTION fn_filter_country (country STRING)
RETURN
    is_account_group_member('admin-sg')
    OR (is_account_group_member('peru-sg') AND country = 'Perú')
    OR (is_account_group_member('usa-sg') AND country = 'USA');

#### Paso 5: Aplicar la Función Row Filter a la Tabla "customers_privacy"
La función Row Filter se asocia directamente a la tabla. Una vez configurada, se ejecuta automáticamente cada vez que se realiza una consulta y devuelve únicamente los registros que cumplen con la condición establecida.

In [0]:
ALTER TABLE customers_privacy
SET ROW FILTER fn_filter_country ON (country);

In [0]:
DESCRIBE EXTENDED customers_privacy;

In [0]:
SELECT * FROM customers_privacy;

#### Paso 6: Crear la Función Column Mask
La función verifica a qué grupo pertenece el usuario y determina si debe mostrar el valor original o una versión enmascarada de la información.

Por ejemplo:
- Los usuarios del grupo Admin pueden visualizar las direcciones de correo electrónico completas.
- Los demás usuarios solo pueden ver una versión enmascarada de las direcciones de correo electrónico.

In [0]:
CREATE OR REPLACE FUNCTION fn_mask_email (email STRING)
RETURN
    CASE 
        WHEN is_account_group_member('admin-sg') THEN email
        ELSE CONCAT(SUBSTR(email, 1, 1), '***@', SPLIT(email, '@')[1])
    END;

In [0]:
CREATE OR REPLACE FUNCTION fn_mask_credit_card (credit_card STRING)
RETURN
    CASE
        WHEN is_account_group_member('admin-sg') THEN credit_card
        ELSE CONCAT(REPEAT('*', LENGTH(credit_card) - 4), RIGHT(credit_card, 4))
    END;

#### Paso 7: Aplicar la Función Mask a la Tabla "customers_privacy"
La función de enmascaramiento se asocia directamente a la columna de correo electrónico y los números de tarjetas de crédito. <p>
Una vez configurada, se ejecuta automáticamente cada vez que se consulta dicha columna.

In [0]:
ALTER TABLE customers_privacy
ALTER COLUMN email SET MASK fn_mask_email;

In [0]:
ALTER TABLE customers_privacy
ALTER COLUMN credit_card SET MASK fn_mask_credit_card;

In [0]:
DESCRIBE EXTENDED customers_privacy;

In [0]:
SELECT * FROM customers_privacy;